# Qdrant + Ollama RAG Flow Playground
This notebook demonstrates the end-to-end RAG cycle:
1. Searching Qdrant database to retrieve relevant context vectors
2. Formatting a context-aware system prompt
3. Generating final assistant completion from the configured LLM reasoning model.

In [1]:
import sys
from pathlib import Path

# Locate and append project root
def get_project_root() -> Path:
    current = Path.cwd()
    for _ in range(5):
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return Path.cwd()

sys.path.append(str(get_project_root().resolve()))

In [2]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
import httpx
import uuid
from src.ragforge.config import QDRANT_URL, OLLAMA_URL, DEFAULT_EMBEDDING_MODEL, DEFAULT_LLM_MODEL, DEFAULT_COLLECTION

client = QdrantClient(url=QDRANT_URL)
test_collection = "notebook-rag-collection"

# 1. Clean up & Create Collection
if client.collection_exists(test_collection):
    client.delete_collection(test_collection)

client.create_collection(
    collection_name=test_collection,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

True

## 2. Seed Mock Document Knowledge Base

In [3]:
def get_embedding(text: str) -> list[float]:
    res = httpx.post(
        f"{OLLAMA_URL}/api/embeddings",
        json={"model": DEFAULT_EMBEDDING_MODEL, "prompt": text},
        timeout=15.0
    )
    return res.json()["embedding"]

kb_docs = [
    "Temporal workers poll task queues and execute workflows sequentially.",
    "Qdrant offers payload index optimization for rapid filtering.",
    "OpenProject enables Agile boards and Scrum backlog tracking."
]

points = []
for doc in kb_docs:
    points.append(PointStruct(
        id=str(uuid.uuid4()),
        vector=get_embedding(doc),
        payload={"text": doc}
    ))

client.upsert(collection_name=test_collection, points=points)
print("Seeded knowledge base!")

Seeded knowledge base!


## 3. Retrieve Context & Query LLM (RAG)

In [4]:
user_query = "How does project management work in OpenProject?"
query_vector = get_embedding(user_query)

# Search vector DB
hits = client.query_points(
    collection_name=test_collection,
    query=query_vector,
    limit=1
).points

context = ""
if hits:
    context = hits[0].payload["text"]
    print(f"Retrieved Context: '{context}'")
else:
    print("No relevant context found.")

# Construct RAG prompt
system_prompt = f"You are a helpful project assistant. Use the following context to answer the question:\nContext: {context}"

# Call LLM
response = httpx.post(
    f"{OLLAMA_URL}/api/chat",
    json={
        "model": DEFAULT_LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        "stream": False
    },
    timeout=45.0
)
response.raise_for_status()
answer = response.json()["message"]["content"]
print(f"\nAssistant Answer:\n{answer}")

Retrieved Context: 'OpenProject enables Agile boards and Scrum backlog tracking.'

Assistant Answer:
Based on the context provided, OpenProject helps with project management by enabling specific methodologies and tools:

*   **Agile Boards:** It provides functionality for Agile boards, allowing teams to visualize their workflow and manage tasks iteratively.
*   **Scrum Backlog Tracking:** It specifically supports Scrum backlog tracking, which is crucial for managing product requirements and planning sprints within the Scrum framework.

In essence, OpenProject equips users with tools tailored for both Agile and Scrum project management styles.


## 4. Clean up

In [5]:
client.delete_collection(test_collection)
print("Cleaned up temporary collection successfully.")

Cleaned up temporary collection successfully.


## 5. Other collections - [history | ingested-docs]

In [29]:
for collection in client.get_collections():
    for a in collection[1]:
        print(f"Collection Name: {a.name}")

Collection Name: chat-history-collection
Collection Name: ragforge-collection
Collection Name: test_ingestion_coll_187959d7
Collection Name: test_ingestion_coll_1b2ba28c
Collection Name: test_ingestion_coll_346a6afa
Collection Name: test_ingestion_coll_3de342cf
Collection Name: test_ingestion_coll_3f896351
Collection Name: test_ingestion_coll_55eae95b
Collection Name: test_ingestion_coll_7524fe07
Collection Name: test_ingestion_coll_7d963529
Collection Name: test_ingestion_coll_801d7ad3
Collection Name: test_ingestion_coll_881d61ba
Collection Name: test_ingestion_coll_9fa89c91
Collection Name: test_ingestion_coll_ad452d5e
Collection Name: test_ingestion_coll_b23f6615
Collection Name: test_ingestion_coll_d0dfbad4
Collection Name: test_ingestion_coll_e5ea0612


In [46]:
def get_all_points_from_collection(collection_name: str="ragforge-collection") -> list[PointStruct]:
    offset = None
    all_points = []

    while True:
        points, next_offset = client.scroll(
            collection_name=collection_name,
            limit=100,
            offset=offset,
            with_payload=True,
            with_vectors=False,
        )

        all_points.extend(points)

        if next_offset is None:
            break

        offset = next_offset
    print(f"Total points retrieved from '{collection_name}': {len(all_points)}")
    return all_points

In [ ]:
raw_points = get_all_points_from_collection("ragforge-collection")
raw_points[0].model_dump()

In [49]:
chat_raw_points = get_all_points_from_collection("chat-history-collection")
chat_raw_points[0].model_dump()

Total points retrieved from 'chat-history-collection': 26


{'id': '0be67206-399d-46b3-a903-1743bf8adb79',
 'payload': {'text': 'User: what are the docs i have ?\nAssistant: I can search through all the documents you have ingested into the knowledge base, but I need a little more information to help you!\n\nCould you tell me what topic or keywords you are interested in? For example, are you looking for documentation related to "project timelines," "API specifications," or "marketing strategy"?',
  'session_id': '51e79c6a-dbbc-4963-be6f-9d3a99f4700a',
  'user_message': 'what are the docs i have ?',
  'assistant_message': 'I can search through all the documents you have ingested into the knowledge base, but I need a little more information to help you!\n\nCould you tell me what topic or keywords you are interested in? For example, are you looking for documentation related to "project timelines," "API specifications," or "marketing strategy"?',
  'timestamp': '2026-06-20T06:06:53.739702'},
 'vector': None,
 'shard_key': None,
 'order_value': None}

In [52]:
# for rec in raw_points:
#     print(rec.payload.get("text"))